In [1]:
from datetime import date
from pathlib import Path

from tapas_gmm.env.rlbench import RLBenchEnvironment, RLBenchEnvironmentConfig
from tapas_gmm.policy.gmm import GMMPolicy, GMMPolicyConfig
from tapas_gmm.policy.models.tpgmm import AutoTPGMMConfig

from rlbench.action_modes.arm_action_modes import BimanualEndEffectorPoseViaIK

import imageio.v2 as imageio

2026-07-20 22:09:51.867 | INFO     |  Running on cpu


/home/nils/Documents/Study Project/Code/riepybdlib/riepybdlib/data.py:34: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_listdir


In [2]:
checkpoint_root = Path("../artifacts/checkpoints/tapas")
checkpoint_dir = max(path for path in checkpoint_root.iterdir() if path.is_dir())
model_path = checkpoint_dir / "latest.pkl"
video_dir = Path("../artifacts/videos/tapas") / date.today().isoformat()
video_dir.mkdir(parents=True, exist_ok=True)

In [3]:
tapas_env = RLBenchEnvironment(
    RLBenchEnvironmentConfig(
        action_mode = BimanualEndEffectorPoseViaIK,
        robot_setup = "dual_panda",
        task = "BimanualDualPushButtons",
        cameras = ("front",),
        camera_pose = {},
        image_size = (128, 128),
        static = False,
        headless = False,
        scale_action = False,
        delay_gripper = False,
        gripper_plot = False,   
    )
)

policy = GMMPolicy(
    GMMPolicyConfig(
        suffix=None,
        model = AutoTPGMMConfig(),
        batch_predict_in_t_models = False,
        topp_in_t_models = False,
        binary_gripper_action = True,
        force_overwrite_checkpoint_config = True,
        pos_lag_thresh=0.03,
        quat_change_thresh=0.1,
    )
)

2026-07-20 22:09:58.413 | INFO     |  Initializing Policy:
2026-07-20 22:09:58.415 | INFO     |  No encoder config provided. Using None.
None


In [4]:
obs = tapas_env.reset()
policy.from_disk(str(model_path))
policy.eval()
policy.reset_episode(tapas_env)

2026-07-20 22:09:59.879 | INFO     |  Loading model:
2026-07-20 22:10:00.033 | ERROR    |  Config mismatch
root.demos_segmentation.components_prop_to_len True != False
root.demos_segmentation.velocity_based True != False
root.demos_segmentation.distance_based False != True
root.frame_selection.rel_score_threshold 0.1 != 0.75
root.tpgmm.reg_init_diag 0.0005 != 5e-05
root.tpgmm.add_action_component False != True
root.tpgmm.reg_diag_gripper 0.02 != 0.1
root.tpgmm.add_time_component True != False
root.tpgmm.reg_diag  0.0002 != 0.001
root.tpgmm.reg_em_finish_diag 0.0002 != 0.001
root.tpgmm.reg_em_finish_diag_gripper 0.02 != 0.1
root.tpgmm.add_gripper_action True != False

2026-07-20 22:10:00.033 | WARNING  |  Overwriting config. This can lead to unexpected errors.
2026-07-20 22:10:00.033 | INFO     |  Detected time-based model: True. Using time-driven policy. Set time_based in config to overwrite.
2026-07-20 22:10:00.034 | INFO     |  Creating local marginals
2026-07-20 22:10:00.034 | INFO 

In [5]:
total_reward = 0
frames = []

for step in range(450):
    action, info = policy.predict(obs)
    obs, reward, done, env_info = tapas_env.step(action)
    frame = obs.cameras["front"].rgb

    if frame.ndim == 4:
        frame = frame[0]

    if frame.shape[0] == 3:
        frame = frame.permute(1, 2, 0)

    frame = (frame * 255).clip(0, 255).byte().cpu().numpy()
    frames.append(frame)
    total_reward += reward

    if obs is None or done or info.get("done", False):
        break

imageio.mimsave(video_dir / "run.mp4", frames, fps=20)

print("steps:", step + 1)
print("total_reward:", total_reward)

tapas_env.close()

2026-07-20 22:10:00.050 | WARNING  |  Implementation lacking modulo rots, enforce z-up/down, etc.
2026-07-20 22:10:00.204 | INFO     |  Action [-3.31225594e-05 -7.84633615e-06  2.91983159e-04 -2.82983507e-05
  2.21002763e-04 -2.66654584e-04  9.99999940e-01  1.00000000e+00
  0.00000000e+00  8.60734935e-04 -1.64918278e-04 -1.43852755e-03
  1.52498639e-04 -4.02995051e-04 -6.69740825e-04  9.99999683e-01
  1.00000000e+00  0.00000000e+00]
2026-07-20 22:10:00.357 | INFO     |  Pos lag: [[ 6.84066909e-04  1.26464415e-04 -8.97796017e-04]
 [-9.74188623e-05 -1.21375202e-05 -1.74996201e-04]], quat lag: 0.0024562329298885395, pos change [[ 4.7722459e-04  1.4066696e-04 -2.9277802e-04]
 [ 1.9966066e-04  4.2915344e-06  4.5049191e-04]], quat change 0.0013810679388648144
2026-07-20 22:10:00.358 | INFO     |  Action [ 5.21881667e-05 -1.19375694e-05 -1.95796041e-04 -1.24820327e-05
 -2.36614327e-04 -1.15618000e-04  9.99999965e-01  1.00000000e+00
  0.00000000e+00  4.55556658e-04 -6.76589840e-05 -1.04364260e

/home/nils/Documents/Study Project/Code/TAPAS/tapas_gmm/utils/geometry_np.py:84: RuntimeWarning: invalid value encountered in arccos
  theta = 2 * np.arccos(2 * np.dot(q, r) ** 2 - 1)


2026-07-20 22:10:00.596 | INFO     |  Pos lag: [[ 4.21178170e-04  6.34065682e-05 -7.75411357e-04]
 [-1.57575922e-04 -1.13224238e-05 -3.88634672e-04]], quat lag: 0.0027653802793963914, pos change [[ 2.6205182e-04  6.4454973e-05 -1.2910366e-04]
 [ 6.0081482e-05 -6.4074993e-07  2.1100044e-04]], quat change 0.0
2026-07-20 22:10:00.597 | INFO     |  Action [ 5.69775633e-05 -1.18590930e-05 -3.90391260e-04 -1.36457757e-05
 -4.61139454e-04 -7.38830829e-05  9.99999891e-01  1.00000000e+00
  0.00000000e+00  2.23734833e-04 -2.58361734e-05 -8.33556644e-04
  2.05159425e-05 -7.37941466e-04 -7.68579839e-05  9.99999725e-01
  1.00000000e+00  0.00000000e+00]
2026-07-20 22:10:00.736 | INFO     |  Pos lag: [[ 3.24344960e-04  1.82380832e-05 -6.99049750e-04]
 [-2.14811832e-04  2.68332178e-06 -4.61825563e-04]], quat lag: 0.0029696992918357597, pos change [[ 8.8095665e-05  4.3809414e-05 -5.6982040e-05]
 [ 6.6295266e-05 -1.4588237e-05  9.6321106e-05]], quat change nan
2026-07-20 22:10:00.737 | INFO     |  Actio

2026-07-20 22:10:00.897 | INFO     |  Pos lag: [[ 2.71989130e-04  2.66843712e-05 -6.03234000e-04]
 [-2.58427135e-04  7.77684295e-07 -5.54415935e-04]], quat lag: 0.002450901159459029, pos change [[ 3.84747982e-05 -1.70618296e-06 -1.15036964e-04]
 [ 5.26756048e-05  1.72853470e-06  1.07645988e-04]], quat change nan
2026-07-20 22:10:00.898 | INFO     |  Action [ 1.14217927e-04 -4.23134867e-06 -5.53798289e-04 -6.41908883e-04
 -6.33241250e-04 -5.17074129e-03  9.99986225e-01  1.00000000e+00
  0.00000000e+00 -2.76941625e-05 -9.27270143e-05 -1.00782947e-03
  3.42144935e-05 -4.83094784e-05  2.34582818e-04  9.99999971e-01
  1.00000000e+00  0.00000000e+00]
2026-07-20 22:10:01.043 | INFO     |  Pos lag: [[ 2.94292610e-04  7.38485230e-05 -7.98542926e-04]
 [-2.49363375e-04  1.94677338e-06 -5.73315163e-04]], quat lag: 0.008696562097664005, pos change [[-8.9496374e-05  3.6790967e-05 -1.8680096e-04]
 [ 6.4373016e-06 -6.3478947e-06  6.2704086e-05]], quat change 0.012352652018538952
2026-07-20 22:10:01.04

2026-07-20 22:10:08.470 | INFO     |  Pos lag: [[-2.68439576e-04 -1.15242447e-04 -1.77322724e-04]
 [ 2.44921754e-04  3.31558355e-06 -4.79282614e-04]], quat lag: 0.0031481454826271247, pos change [[ 2.1457672e-06 -3.9801002e-05  1.9603968e-04]
 [ 7.0661306e-05  6.6667795e-05  5.6529045e-04]], quat change 0.0013810679388648144
2026-07-20 22:10:08.471 | INFO     |  Action [ 1.02981976e-04 -2.43175292e-04  7.79521667e-05 -6.95430658e-04
 -2.60770827e-04  2.03840416e-04 -9.99999703e-01  1.00000000e+00
  0.00000000e+00 -2.19066421e-04  1.71538981e-04  6.08618893e-05
 -9.52023849e-05 -2.93850759e-04  4.13255961e-05  9.99999951e-01
  0.00000000e+00  0.00000000e+00]
2026-07-20 22:10:08.648 | WARNING  |  Product did not converge in 50 iterations.
2026-07-20 22:10:08.892 | INFO     |  Pos lag: [[-2.98043115e-04 -1.28309054e-04 -2.68209309e-04]
 [ 2.39887638e-04 -1.76184697e-05 -4.60836236e-04]], quat lag: 0.002912040847547105, pos change [[ 7.9125166e-05 -4.3332577e-05  3.2931566e-04]
 [ 2.796948

2026-07-20 22:10:09.600 | INFO     |  Pos lag: [[-0.07094595 -0.04571228 -0.00120161]
 [ 0.06340271 -0.01253174  0.21230705]], quat lag: 4.6531499991497745, pos change [[ 6.6261441e-03 -5.7841510e-02  8.8991642e-02]
 [ 2.4542212e-05 -1.6093254e-06  2.8312206e-04]], quat change 1.6423891531166852
2026-07-20 22:10:09.601 | INFO     |  Action [ 0.0589468  -0.02781425  0.26946963 -0.23396595 -0.63082478  0.06287308
 -0.73713432  1.          0.         -0.07530675 -0.0047076  -0.03293132
  0.00552993 -0.18058424  0.001111    0.98354335  0.          0.        ]


2026-07-20 22:10:09.992 | INFO     |  Pos lag: [[-0.0679493  -0.04519063  0.00792636]
 [ 0.06558304 -0.017576    0.26873885]], quat lag: 2.968060821393942, pos change [[-2.99304724e-04 -1.42872334e-04  1.19924545e-04]
 [ 1.55866146e-05  2.98023224e-06  5.51939011e-05]], quat change 0.03619885983863339
2026-07-20 22:10:09.992 | INFO     |  Action [ 0.05954742 -0.02575703  0.2742567  -0.17077864 -0.51240011  0.07243235
 -0.83847143  1.          0.         -0.07571612 -0.00295869 -0.03029334
  0.00866083 -0.17962715 -0.00742768  0.9836686   0.          0.        ]
2026-07-20 22:10:10.184 | INFO     |  Pos lag: [[-0.06716215 -0.04480853  0.01048213]
 [ 0.06420893 -0.01813275  0.27389497]], quat lag: 2.3050953692698424, pos change [[-5.6937337e-05 -2.8419495e-04 -1.0859966e-04]
 [-3.7491322e-05 -2.0414591e-06 -7.1644783e-05]], quat change 0.023840964226035082
2026-07-20 22:10:10.186 | INFO     |  Action [ 6.19664930e-02 -1.98812781e-02  2.91474406e-01  2.48371233e-02
 -1.18096621e-01  8.776

In [6]:
tapas_env.close()